# Importamos los Paths de los datasets a usar:

In [1]:
import csv
import sys
from pathlib import Path
sys.path.append(str(Path('.').resolve().parent.parent))
from modules.paths import CONNECTIONS_DATA_MODIFIED, AR_DATA

### 9: Mostrar los diferentes tipos de conectividades.

In [2]:
with CONNECTIONS_DATA_MODIFIED.open(mode='r', encoding='UTF-8') as file:
    reader = csv.reader(file)
    types = []
    header = next(reader)

    types = [header[i] for i in range(4, 13)]
    
    print(f"Tipos de conectividad: {types}")

Tipos de conectividad: ['ADSL', 'CABLEMODEM', 'DIALUP', 'FIBRAOPTICA', 'SATELITAL', 'WIRELESS', 'TELEFONIAFIJA', '3G', '4G']


### 10: Mostrar las cantidades de localidades con cada tipo de conectividad.

In [3]:
with CONNECTIONS_DATA_MODIFIED.open(mode='r', encoding='UTF-8') as file:
    connectivity_count = {}
    reader = csv.DictReader(file)

    for row in reader:
        for type in types:
            if row[type] == 'SI':
                connectivity_count[type] = connectivity_count.get(type, 0) + 1
                
for type, count in connectivity_count.items():
   print(f"{type} : Cantidad de localidades: {count}")

ADSL : Cantidad de localidades: 1149
CABLEMODEM : Cantidad de localidades: 851
FIBRAOPTICA : Cantidad de localidades: 1321
SATELITAL : Cantidad de localidades: 998
WIRELESS : Cantidad de localidades: 2145
TELEFONIAFIJA : Cantidad de localidades: 2086
3G : Cantidad de localidades: 1880
4G : Cantidad de localidades: 2574
DIALUP : Cantidad de localidades: 396


### 11: Mostrar las provincias para las cuales todas sus ciudades poseen FIBRA ÓPTICA.

In [4]:
with CONNECTIONS_DATA_MODIFIED.open(mode='r', encoding='UTF-8') as file:
    provinces_fiber_optics = {} #--> guardamos las provincias que todas sus ciudades tengan fibra optica
    reader = csv.DictReader(file)

    for row in reader:
        get_province = row['Provincia']
        if get_province not in provinces_fiber_optics:  #guardamos la provincia si aun no a sido guardada.
            provinces_fiber_optics[get_province] = True
        if row['FIBRAOPTICA'] == 'NO':                  #modificamos su valor en False si NO posee fibra optica.
            provinces_fiber_optics[get_province] = False

for prov, condition in provinces_fiber_optics.items():
    if condition:
        print(f"Todas las ciudades de <{prov}> poseen Fibra óptica")

Todas las ciudades de <CABA> poseen Fibra óptica


### 12: Mostrar para cada provincia su capital y, si se conoce la información para dicha capital, informar si posee conectividad (campo 'posee_conectividad' creado previamente). En caso de no conocer la información mostrar el texto “conectividad desconocida”.

Defino una funcion encargada de limpiar una palabra de acentos:

In [5]:
def remove_accents(expression):
    accents = {'á' : 'a', 'é' : 'e', 'í' : 'i', 'ó' : 'o', 'ú' : 'u', 'Á' : 'A', 'É' : 'E', 'Í' : 'I', 'Ó' : 'O', 'Ú' : 'U'}

    clean_expression = ''.join(accents.get(letter, letter) for letter in expression) #si letter esta en accents retorna su valor, si no esta retorna la letra sin cambios.
    return clean_expression

Funcion que limpia una palabra que contenga contenido entre "(...)" , separa por comas y remplaza los acentos llamando a 'remove_accents':

In [6]:
import re
def clean_word(word):
    term = re.sub(r'\([^)]*\)', '', word).strip() # re.sub es una funcion encargada realizar sustituciones de patrones en cadenas de texto, 
    if ',' in term:                               # en este caso elimina lo que encuentre entre "(__)" incluyendolos.
        term = term.split(',')[0]
    term = remove_accents(term)
    return term

Obtengo la capital y su provincia:

In [7]:
cap_data = {} #--> contiene la capital y sus respectivos datos (provincia y si posee conectividad)

with AR_DATA.open(mode='r', encoding='UTF-8') as csv_ar:
    ar_reader = csv.DictReader(csv_ar)
    
    for row_ar in ar_reader:

        p_province = clean_word(row_ar['admin_name'])
        is_capital = row_ar['capital']

        if is_capital == 'admin':           #si is_capital == 'admin' significa que es una capital y entonces la guardamos en cap_data.
            
            city = clean_word(row_ar['city'].lower())   #limpiamos 'ciudad'.
            data = [p_province, 'Conectividad Desconocida']
            cap_data[city] = data


Funcion que actualiza nuestro diccionario segun la columna Posee_Conectividad:

In [8]:
def update_connectivity_info(disctric, has_connect):

    if cap_data[disctric.lower()][1] == 'Conectividad Desconocida':
        cap_data[disctric.lower()][1] = 'NO'
    if has_connect == 'SI':
        cap_data[disctric.lower()][1] = has_connect

In [9]:
with CONNECTIONS_DATA_MODIFIED.open(mode='r', encoding='UTF-8') as file:
    reader = csv.DictReader(file)

    capitals = list(cap_data.keys())

    for row in reader:

        has_connectivity = row['Posee_Conectividad']
        locality = clean_word(row['Localidad'])
        province = clean_word(row['Provincia'])

        if locality.lower() in cap_data:
            
            province_origin = str(cap_data[locality.lower()][0]) #-->cap_data[0] = provincia 
                                                                #-->cap_data[1] = posee_conectividad o conectividad desconocida

            if province_origin.lower() == province.lower(): #si las provincias coinciden encontramos la capital, entonces actualizamos los datos de cap_data.
                     
                update_connectivity_info(locality, has_connectivity)
                

for capital, datos in cap_data.items(): #mostramos Provincia - Capital - Posee o no conectividad.
    cad = f"Provincia: {datos[0]} ---|--- Capital: {capital} ----|--- "
    if datos[1] == 'Conectividad Desconocida':
        cad += datos[1]
    else:
        cad += f"{datos[1]} Posee Conectividad"
    print(cad)


Provincia: Cordoba ---|--- Capital: cordoba ----|--- SI Posee Conectividad
Provincia: Santiago del Estero ---|--- Capital: santiago del estero ----|--- SI Posee Conectividad
Provincia: Tucuman ---|--- Capital: san miguel de tucuman ----|--- SI Posee Conectividad
Provincia: Salta ---|--- Capital: salta ----|--- SI Posee Conectividad
Provincia: San Juan ---|--- Capital: san juan ----|--- SI Posee Conectividad
Provincia: Santa Fe ---|--- Capital: santa fe ----|--- SI Posee Conectividad
Provincia: Corrientes ---|--- Capital: corrientes ----|--- SI Posee Conectividad
Provincia: Jujuy ---|--- Capital: san salvador de jujuy ----|--- SI Posee Conectividad
Provincia: Chaco ---|--- Capital: resistencia ----|--- SI Posee Conectividad
Provincia: Misiones ---|--- Capital: posadas ----|--- SI Posee Conectividad
Provincia: Entre Rios ---|--- Capital: parana ----|--- SI Posee Conectividad
Provincia: Formosa ---|--- Capital: formosa ----|--- SI Posee Conectividad
Provincia: Neuquen ---|--- Capital: neu